<a href="https://colab.research.google.com/github/tustus1022-ui/esaa/blob/main/%EC%B5%9C%EC%A2%85_%EC%9E%AC%ED%95%99%EC%8A%B5%EC%BD%94%EB%93%9C%2B%EB%AA%A8%EB%8D%B8%ED%8C%8C%EC%9D%BC_%EC%A0%80%EC%9E%A5%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install numerapi -q
import numerapi
import getpass

NUMERAI_PUBLIC_ID = getpass.getpass("Public ID 입력: ").strip()
NUMERAI_SECRET_KEY = getpass.getpass("Secret Key 입력: ").strip()

napi = numerapi.NumerAPI(public_id=NUMERAI_PUBLIC_ID, secret_key=NUMERAI_SECRET_KEY)
print(napi.get_models())

model_id = napi.get_models()["esaa_maddox"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Public ID 입력: ··········
Secret Key 입력: ··········
{'esaa_maddox': '37dd8d41-54d3-4e4a-a6e8-7b5c7a49ad0f', 'esaa': 'b334bb14-b052-444e-a266-7bb8619e749f'}


In [ ]:
import os
os.makedirs('/content/numerai_data', exist_ok=True)

if not os.path.exists("/content/numerai_data/train.parquet"):
    napi.download_dataset("v5.2/train.parquet", "/content/numerai_data/train.parquet")

if not os.path.exists("/content/numerai_data/validation.parquet"):
    napi.download_dataset("v5.2/validation.parquet", "/content/numerai_data/validation.parquet")

print("다운로드 완료")

/content/numerai_data/train.parquet: 100%|██████████| 2.60G/2.60G [00:55<00:00, 46.5MB/s]
/content/numerai_data/validation.parquet: 100%|██████████| 4.34G/4.34G [02:55<00:00, 24.7MB/s]

다운로드 완료


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SAVE_DIR = "/content/drive/MyDrive/ESAA/numerai_postprocess"

with open(f"{SAVE_DIR}/selected_features.json") as f:
    selected_features = json.load(f)

scaler = joblib.load(f"{SAVE_DIR}/fitted_scaler.joblib")
pca = joblib.load(f"{SAVE_DIR}/fitted_pca.joblib")

def iter_parquet_batches(path, columns, batch_size):
    pf = pq.ParquetFile(path)
    for batch in pf.iter_batches(columns=columns, batch_size=batch_size):
        chunk = batch.to_pandas()
        if chunk.index.name is not None or "id" not in chunk.columns:
            chunk = chunk.reset_index()
        yield chunk

def clean_array(arr):
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def numerai_corr_arrays(era_arr, target_arr, pred_arr):
    df_eval = pd.DataFrame({"era": era_arr, "target": target_arr, "prediction": pred_arr})
    def era_corr(sub):
        return np.corrcoef(sub["prediction"].rank(pct=True), sub["target"])[0, 1]
    return df_eval.groupby("era").apply(era_corr, include_groups=False)

In [ ]:
TRAIN_PATH = "/content/numerai_data/train.parquet"
BATCH_SIZE = 100_000

X_parts, era_parts, id_parts, target_parts = [], [], [], []
train_columns = ["id", "era", "target"] + selected_features

for chunk in iter_parquet_batches(TRAIN_PATH, train_columns, BATCH_SIZE):
    X = clean_array(chunk[selected_features].to_numpy(dtype=np.float32))
    X_scaled = scaler.transform(X)
    X_pca = pca.transform(X_scaled).astype(np.float32)
    X_parts.append(X_pca)
    era_parts.append(chunk["era"].to_numpy())
    id_parts.append(chunk["id"].to_numpy())
    target_parts.append(chunk["target"].to_numpy())
    print(f"처리: {sum(len(p) for p in id_parts)} rows")

pca_cols = [f"pca_{i}" for i in range(pca.n_components_)]
train_pca_df = pd.DataFrame(np.vstack(X_parts), columns=pca_cols)
train_pca_df["era"] = np.concatenate(era_parts)
train_pca_df["id"] = np.concatenate(id_parts)
train_pca_df["target"] = np.concatenate(target_parts)

LAGS = (1, 2)
def add_era_level_lag_features(df, pca_cols, lags=LAGS):
    era_mean = df.groupby("era")[pca_cols].mean().sort_index()
    lag_frames = []
    for lag in lags:
        shifted = era_mean.shift(lag)
        shifted.columns = [f"{c}_lag{lag}" for c in pca_cols]
        lag_frames.append(shifted)
    era_lag_features = pd.concat(lag_frames, axis=1)
    return df.merge(era_lag_features, on="era", how="left")

train_final_real = add_era_level_lag_features(train_pca_df, pca_cols)
lag_cols = [c for c in train_final_real.columns if "_lag" in c]
train_final_real[lag_cols] = train_final_real[lag_cols].fillna(0)

print("train_final_real shape:", train_final_real.shape)

처리: 100000 rows
처리: 200000 rows
처리: 300000 rows
처리: 400000 rows
처리: 500000 rows
처리: 600000 rows
처리: 700000 rows
처리: 800000 rows
처리: 900000 rows
처리: 1000000 rows
처리: 1100000 rows
처리: 1200000 rows
처리: 1300000 rows
처리: 1400000 rows
처리: 1500000 rows
처리: 1600000 rows
처리: 1700000 rows
처리: 1800000 rows
처리: 1900000 rows
처리: 2000000 rows
처리: 2100000 rows
처리: 2200000 rows
처리: 2300000 rows
처리: 2400000 rows
처리: 2500000 rows
처리: 2600000 rows
처리: 2700000 rows
처리: 2746268 rows
train_final_real shape: (2746268, 1503)


In [ ]:
VAL_PATH = "/content/numerai_data/validation.parquet"

X_parts, era_parts, id_parts = [], [], []
val_columns = ["id", "era"] + selected_features

for chunk in iter_parquet_batches(VAL_PATH, val_columns, BATCH_SIZE):
    X = clean_array(chunk[selected_features].to_numpy(dtype=np.float32))
    X_scaled = scaler.transform(X)
    X_pca = pca.transform(X_scaled).astype(np.float32)
    X_parts.append(X_pca)
    era_parts.append(chunk["era"].to_numpy())
    id_parts.append(chunk["id"].to_numpy())
    print(f"처리: {sum(len(p) for p in id_parts)} rows")

val_pca_df = pd.DataFrame(np.vstack(X_parts), columns=pca_cols)
val_pca_df["era"] = np.concatenate(era_parts)
val_pca_df["id"] = np.concatenate(id_parts)

val_final = add_era_level_lag_features(val_pca_df, pca_cols)
val_final[lag_cols] = val_final[lag_cols].fillna(0)

print("val_final shape:", val_final.shape)

처리: 100000 rows
처리: 200000 rows
처리: 300000 rows
처리: 400000 rows
처리: 500000 rows
처리: 600000 rows
처리: 700000 rows
처리: 800000 rows
처리: 900000 rows
처리: 1000000 rows
처리: 1100000 rows
처리: 1200000 rows
처리: 1300000 rows
처리: 1400000 rows
처리: 1500000 rows
처리: 1600000 rows
처리: 1700000 rows
처리: 1800000 rows
처리: 1900000 rows
처리: 2000000 rows
처리: 2100000 rows
처리: 2200000 rows
처리: 2300000 rows
처리: 2400000 rows
처리: 2500000 rows
처리: 2600000 rows
처리: 2700000 rows
처리: 2800000 rows
처리: 2900000 rows
처리: 3000000 rows
처리: 3100000 rows
처리: 3200000 rows
처리: 3300000 rows
처리: 3400000 rows
처리: 3500000 rows
처리: 3600000 rows
처리: 3700000 rows
처리: 3800000 rows
처리: 3900000 rows
처리: 4000000 rows
처리: 4099982 rows
val_final shape: (4099982, 1502)


In [ ]:
FEATURES_TRAIN = [c for c in train_final_real.columns if c.startswith('pca_')]
FEATURES = [c for c in val_final.columns if c.startswith('pca_')]   # ← 이 줄 추가

X_full = train_final_real[FEATURES_TRAIN]
y_full = train_final_real['target']

In [ ]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 32.9 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive
import pandas as pd

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


여기서부터 과거cell8

In [ ]:
from catboost import CatBoostRegressor

catboost_full = CatBoostRegressor(
    iterations=2000, learning_rate=0.0103, depth=4,
    l2_leaf_reg=2.92, bagging_temperature=0.997,
    random_strength=9.83, border_count=40,
    loss_function='RMSE', eval_metric='RMSE',
    random_seed=42, task_type='GPU', verbose=200,
)
catboost_full.fit(X_full, y_full)  # early stopping용 eval_set 없음 (전체 데이터라 검증셋 없이 학습)

val_final['prediction'] = catboost_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    "/content/drive/MyDrive/ESAA/numerai_postprocess/catboost_fullrefit_diagnostics.csv", index=False)

0:	learn: 0.2236926	total: 7.45s	remaining: 4h 8m 7s
200:	learn: 0.2236609	total: 11s	remaining: 1m 38s
400:	learn: 0.2236362	total: 14.6s	remaining: 58.2s
600:	learn: 0.2236154	total: 18.1s	remaining: 42.1s
800:	learn: 0.2235955	total: 21.6s	remaining: 32.4s
1000:	learn: 0.2235774	total: 25s	remaining: 25s
1200:	learn: 0.2235601	total: 28.4s	remaining: 18.9s
1400:	learn: 0.2235429	total: 31.9s	remaining: 13.6s
1600:	learn: 0.2235241	total: 35.4s	remaining: 8.82s
1800:	learn: 0.2235019	total: 39s	remaining: 4.31s
1999:	learn: 0.2234791	total: 42.7s	remaining: 0us


/tmp/ipykernel_2151/1191495201.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_final['prediction'] = catboost_full.predict(val_final[FEATURES])


In [ ]:
import xgboost as xgb

xgb_full = xgb.XGBRegressor(
    learning_rate=0.01683695685806219, max_depth=3,
    subsample=0.6842255443980394, colsample_bytree=0.9160641130149365,
    reg_alpha=0.27188026107491425, reg_lambda=6.848672546896612,
    min_child_weight=7, n_estimators=2000,
    eval_metric='rmse', random_state=42, device='cuda',
)
xgb_full.fit(X_full, y_full)  # early_stopping_rounds 제거 (eval_set 없으니)

val_final['prediction'] = xgb_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    "/content/drive/MyDrive/ESAA/numerai_postprocess/xgboost_fullrefit_diagnostics.csv", index=False)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [03:30:52] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
import lightgbm as lgb

lgb_full = lgb.LGBMRegressor(
    num_leaves=188, max_depth=11, learning_rate=0.006277067000913665,
    n_estimators=1098, min_child_samples=41, subsample=0.7408435028494748,
    subsample_freq=1, colsample_bytree=0.5101189478563521,
    reg_alpha=0.004623707168862111, reg_lambda=3.2967598858203895e-06,
    objective='regression', metric='rmse', verbosity=-1,
    random_state=42, n_jobs=-1,
)
lgb_full.fit(X_full, y_full)

val_final['prediction'] = lgb_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    "/content/drive/MyDrive/ESAA/numerai_postprocess/lightgbm_fullrefit_diagnostics.csv", index=False)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_full = RandomForestRegressor(
    n_estimators=265, max_depth=5, min_samples_leaf=163,
    max_features=0.1771710272538529, n_jobs=-1, random_state=42,
)
rf_full.fit(X_full, y_full)

val_final['prediction'] = rf_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    "/content/drive/MyDrive/ESAA/numerai_postprocess/rf_fullrefit_diagnostics.csv", index=False)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et_full = ExtraTreesRegressor(
    n_estimators=140, max_depth=5, min_samples_leaf=125,
    max_features=0.3098554283349224, n_jobs=-1, random_state=42,
)
et_full.fit(X_full, y_full)

val_final['prediction'] = et_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    "/content/drive/MyDrive/ESAA/numerai_postprocess/et_fullrefit_diagnostics.csv", index=False)

In [ ]:
import pandas as pd
check = pd.read_csv(f"/content/drive/MyDrive/ESAA/numerai_postprocess/catboost_fullrefit_diagnostics.csv")
print(check.columns.tolist())   # ['id', 'prediction'] 나와야 함
print(check.shape)              # (4050241, 2) 근처 나와야 함 (지난번 validation 행 수)
print(check['prediction'].between(0, 1).all())  # True 나와야 함 (numerai 제출 조건)

['id', 'prediction']
(4085748, 2)
True


In [ ]:
import os
os.makedirs(f"{SAVE_DIR}/models", exist_ok=True)

여기서부터 과거 cell 8-12의 수정본

In [ ]:
#============================================
# cell 8 수정본 - CatBoost (모델 저장 추가)
# ============================================
from catboost import CatBoostRegressor

catboost_full = CatBoostRegressor(
    iterations=2000, learning_rate=0.0103, depth=4,
    l2_leaf_reg=2.92, bagging_temperature=0.997,
    random_strength=9.83, border_count=40,
    loss_function='RMSE', eval_metric='RMSE',
    random_seed=42, task_type='GPU', verbose=200,
)
catboost_full.fit(X_full, y_full)

# --- 모델 저장 (추가된 부분) ---
catboost_full.save_model(f"{SAVE_DIR}/models/catboost_model.cbm")
print("catboost 모델 저장 완료")

val_final['prediction'] = catboost_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    f"{SAVE_DIR}/catboost_fullrefit_diagnostics.csv", index=False)


0:	learn: 0.2236926	total: 20.4ms	remaining: 40.8s
200:	learn: 0.2236609	total: 3.6s	remaining: 32.2s
400:	learn: 0.2236362	total: 7.12s	remaining: 28.4s
600:	learn: 0.2236154	total: 10.6s	remaining: 24.6s
800:	learn: 0.2235955	total: 14.1s	remaining: 21.1s
1000:	learn: 0.2235774	total: 17.5s	remaining: 17.4s
1200:	learn: 0.2235601	total: 20.9s	remaining: 13.9s
1400:	learn: 0.2235429	total: 24.3s	remaining: 10.4s
1600:	learn: 0.2235241	total: 27.7s	remaining: 6.91s
1800:	learn: 0.2235019	total: 31.3s	remaining: 3.46s
1999:	learn: 0.2234791	total: 34.9s	remaining: 0us
catboost 모델 저장 완료


/tmp/ipykernel_2554/3884191743.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_final['prediction'] = catboost_full.predict(val_final[FEATURES])


In [ ]:
# ============================================
# cell 9 수정본 - XGBoost (모델 저장 추가)
# ============================================
import xgboost as xgb

xgb_full = xgb.XGBRegressor(
    learning_rate=0.01683695685806219, max_depth=3,
    subsample=0.6842255443980394, colsample_bytree=0.9160641130149365,
    reg_alpha=0.27188026107491425, reg_lambda=6.848672546896612,
    min_child_weight=7, n_estimators=2000,
    eval_metric='rmse', random_state=42, device='cuda',
)
xgb_full.fit(X_full, y_full)

# --- 모델 저장 (추가된 부분) ---
# 주의: xgb.save_model()은 트리 구조만 저장하고 sklearn wrapper 하이퍼파라미터는
# 복원 안 되는 버그가 있었으니(지난번에 겪은 문제), 파라미터도 json으로 같이 저장
xgb_full.save_model(f"{SAVE_DIR}/models/xgboost_model.json")
import json as _json
with open(f"{SAVE_DIR}/models/xgboost_params.json", "w") as f:
    _json.dump(xgb_full.get_params(), f)
print("xgboost 모델 + 파라미터 저장 완료")

val_final['prediction'] = xgb_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    f"{SAVE_DIR}/xgboost_fullrefit_diagnostics.csv", index=False)


xgboost 모델 + 파라미터 저장 완료


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:553: UserWarning: [13:43:31] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
#============================================
# cell 10 수정본 - LightGBM (모델 저장 추가)
# ============================================
import lightgbm as lgb

lgb_full = lgb.LGBMRegressor(
    num_leaves=188, max_depth=11, learning_rate=0.006277067000913665,
    n_estimators=1098, min_child_samples=41, subsample=0.7408435028494748,
    subsample_freq=1, colsample_bytree=0.5101189478563521,
    reg_alpha=0.004623707168862111, reg_lambda=3.2967598858203895e-06,
    objective='regression', metric='rmse', verbosity=-1,
    random_state=42, n_jobs=-1,
)
lgb_full.fit(X_full, y_full)

# --- 모델 저장 (추가된 부분) ---
lgb_full.booster_.save_model(f"{SAVE_DIR}/models/lightgbm_model.txt")
print("lightgbm 모델 저장 완료")

val_final['prediction'] = lgb_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    f"{SAVE_DIR}/lightgbm_fullrefit_diagnostics.csv", index=False)


lightgbm 모델 저장 완료


In [ ]:
# ============================================
# cell 11 수정본 - RandomForest (모델 저장 추가)
# ============================================
from sklearn.ensemble import RandomForestRegressor
import joblib

rf_full = RandomForestRegressor(
    n_estimators=265, max_depth=5, min_samples_leaf=163,
    max_features=0.1771710272538529, n_jobs=-1, random_state=42,
)
rf_full.fit(X_full, y_full)

# --- 모델 저장 (추가된 부분) ---
joblib.dump(rf_full, f"{SAVE_DIR}/models/rf_model.joblib")
print("RF 모델 저장 완료")

val_final['prediction'] = rf_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    f"{SAVE_DIR}/rf_fullrefit_diagnostics.csv", index=False)


RF 모델 저장 완료


In [ ]:
# ============================================
# cell 12 수정본 - ExtraTrees (모델 저장 추가)
# ============================================
from sklearn.ensemble import ExtraTreesRegressor

et_full = ExtraTreesRegressor(
    n_estimators=140, max_depth=5, min_samples_leaf=125,
    max_features=0.3098554283349224, n_jobs=-1, random_state=42,
)
et_full.fit(X_full, y_full)

# --- 모델 저장 (추가된 부분) ---
joblib.dump(et_full, f"{SAVE_DIR}/models/et_model.joblib")
print("ET 모델 저장 완료")

val_final['prediction'] = et_full.predict(val_final[FEATURES])
val_final[['id', 'prediction']].to_csv(
    f"{SAVE_DIR}/et_fullrefit_diagnostics.csv", index=False)

ET 모델 저장 완료
